# 01 — INQUIRE-Rerank data and frozen embeddings

**Goal:** use the benchmark's fixed candidate lists and persist model embeddings once per query. Later notebooks load the cache and only run ranking algorithms, avoiding repeated model inference.

**License warning:** the Hugging Face dataset card currently lists **CC BY-NC 4.0**. Check the current license before downloading or using the data; do not use it for commercial purposes unless separately authorized. The dataset is natural-world/ecology focused, not a fashion benchmark.

Start small with three to five queries. Streaming avoids eagerly building a giant in-memory dataset, but a Hugging Face parquet/image shard may still need to be fetched. The notebook's model uses frozen weights and does not train anything.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src" / "image_rerank").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from image_rerank.data import iter_inquire_queries
from image_rerank.embeddings import EncoderSettings, build_encoder
from image_rerank.experiment import prepare_cached_embeddings
from image_rerank.artifacts import ensure_dir

SPLIT = "validation"   # Use validation for iteration; save test for the locked evaluation.
MAX_QUERIES = 3        # Change to None for the full available split.
BACKEND = "open_clip"  # Change to fashionclip for an out-of-domain diagnostic on INQUIRE.
BATCH_SIZE = 8         # Conservative default for a 4 GB GPU.
CACHE_ROOT = ROOT / "artifacts" / "cache"
ensure_dir(CACHE_ROOT)

print({"split": SPLIT, "max_queries": MAX_QUERIES, "backend": BACKEND, "cache_root": str(CACHE_ROOT)})

## 1. Initialize a frozen encoder

The default is OpenCLIP's ViT-B/32 with the OpenAI pretrained weights. The encoder wrapper chooses CUDA when it is available, uses `torch.inference_mode()`, normalizes embeddings, and uses conservative batches. Use the FashionCLIP backend only when you explicitly want to test its domain-specific representation.

In [ ]:
settings = EncoderSettings(
    backend=BACKEND,
    model_name="ViT-B-32" if BACKEND == "open_clip" else "patrickjohncyh/fashion-clip",
    pretrained="openai",
    device="auto",
    batch_size=BATCH_SIZE,
    use_amp=True,
)
encoder = build_encoder(settings)
print("Encoder cache key:", encoder.cache_key)
print("Resolved device:", encoder.device)

## 2. Load one query shortlist at a time and cache its embeddings

Each cache file contains the query vector, candidate image vectors, image IDs, original CLIP scores, relevance labels, and optional metadata. Vectors are stored as float16 to reduce disk use, then restored to float32 for metrics and similarity calculations.

This cell can be re-run; existing query caches are reused. The loader uses the official Hugging Face dataset ID `evendrow/INQUIRE-Rerank`.

In [ ]:
records = iter_inquire_queries(
    split=SPLIT,
    max_queries=MAX_QUERIES,
    dataset_id="evendrow/INQUIRE-Rerank",
    streaming=True,
)
cache_paths, cache_stats = prepare_cached_embeddings(
    records,
    encoder,
    split=SPLIT,
    cache_root=CACHE_ROOT,
)

print("Cache statistics:", cache_stats)
print("Saved cache files:", len(cache_paths))
display(pd.DataFrame({"cache_path": [str(p.relative_to(ROOT)) for p in cache_paths]}))

## 3. Verify the persisted cache

A small cache is sufficient to validate the next notebook. For reproducible benchmark results, record the encoder key and split and do not mix model backends in the same run.

In [ ]:
from image_rerank.artifacts import load_query_embeddings

first_cache = load_query_embeddings(cache_paths[0])
print("Query:", first_cache.query)
print("Candidates:", first_cache.image_embeddings.shape[0])
print("Embedding dimension:", first_cache.image_embeddings.shape[1])
print("Relevance positives in shortlist:", int(first_cache.relevance.sum()))
print("Original score range:", float(first_cache.original_scores.min()), float(first_cache.original_scores.max()))
print("Encoder cosine score range:", float(first_cache.encoder_scores.min()), float(first_cache.encoder_scores.max()))
assert first_cache.image_embeddings.ndim == 2
assert first_cache.query_embedding.shape[0] == first_cache.image_embeddings.shape[1]

## Artifacts written

`artifacts/cache/<split>/<encoder-key>/<query-id>.npz`

These files are git-ignored. They are reusable across ranking experiments and notebook restarts. For test-set work, only run this notebook after the method is fixed, then avoid using test labels to choose weights or controller settings.